# 02 — Preprocessing for Modelling

**Input:** `data/processed/eda_cleaned.csv` (1,109 rows, 6 target classes, 9 columns)

**Steps:**
1. Minimal text cleaning (remove URLs, email addresses — keep casing & punctuation for transformers)
2. One-hot encode metadata features (`item_type`, `org_broad_category`)
3. Stratified train/val split (85/15)
4. Save train/val CSVs to `data/modelling/`

**Minimal text cleaning:**
I only remove noise that no model should see: URLs and email addresses.

Everything else (casing, punctuation, stopwords) is kept intact because:
- **TF-IDF** handles its own lowercasing and stopword removal inside `TfidfVectorizer`
- **Transformers** need casing and punctuation — heavy cleaning would hurt performance
- **Claude API** works best with natural text

This way the same `text_clean` column works as input for every model.


In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from pathlib import Path

In [2]:
DATA_DIR = Path("../data")
INPUT_PATH = DATA_DIR / "processed" / "eda_cleaned.csv"
OUTPUT_DIR = DATA_DIR / "modelling"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
VAL_SIZE = 0.15

## 1. Load data

In [3]:
df = pd.read_csv(INPUT_PATH)
print(f"Shape: {df.shape}")
print(f"\nTarget distribution:\n{df['target'].value_counts()}")
df.head()

Shape: (1109, 9)

Target distribution:
target
political_environment_key_organisations    239
what_matters_ed                            197
teacher_rrd                                194
edtech                                     175
policy_practice_research                   160
four_nations                               144
Name: count, dtype: int64


,id,newsletter_number,issue_date,link,target,text,organisation,org_broad_category,item_type
0,a65013c4-ce18-43d8-82a5-b7707d8ecce9,1,11 July 2023,https://schoolsweek.co.uk/reject-fewer-teacher...,political_environment_key_organisations,"Reject fewer teacher applicants, DfE tells tra...",schools_week,media_sector,news_article
1,550a3ffb-c512-4117-b6fa-eff93855fd89,1,11 July 2023,https://schoolsweek.co.uk/revealed-the-experts...,political_environment_key_organisations,Revealed: the experts advising ministers on te...,schools_week,media_sector,news_article
2,c79baa9f-775b-444e-bbd4-dc111e4846f4,1,11 July 2023,https://schoolsweek.co.uk/chatgpt-keegan-launc...,policy_practice_research,Deadline 23 August 2023 Education secretary Gi...,schools_week,media_sector,news_article
3,9ec40ab1-2d2e-460b-8c82-1ed4a121d770,1,11 July 2023,https://schoolsweek.co.uk/teachers-back-digita...,political_environment_key_organisations,Ofqual and DfE studying 'feasibility' of 'full...,schools_week,media_sector,news_article
4,4776c84a-705e-4c49-aa46-8c759d90540c,1,11 July 2023,https://schoolsweek.co.uk/revealed-the-full-de...,political_environment_key_organisations,Labour Revealed: The full details of Labour's ...,schools_week,media_sector,news_article


## 2. Minimal text cleaning

I keep casing and punctuation intact — transformer models (steps 3–5) benefit from them.  
Only remove noise that no model should see: URLs and email addresses.

In [4]:
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+")
EMAIL_PATTERN = re.compile(r"\S+@\S+\.\S+")


def clean_text(text: str) -> str:
    """Remove URLs and email addresses, collapse whitespace."""
    text = URL_PATTERN.sub("", text)
    text = EMAIL_PATTERN.sub("", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


df["text_clean"] = df["text"].astype(str).apply(clean_text)

In [8]:
# Spot-check: show rows where cleaning made a difference
changed = df[df["text"] != df["text_clean"]]
print(f"{len(changed)} rows changed by text cleaning\n")

for _, row in changed.head(-5).iterrows():
    print("BEFORE:", row["text"][:120])
    print("AFTER: ", row["text_clean"][:120])
    print()

162 rows changed by text cleaning

BEFORE: Ofqual and DfE studying 'feasibility' of 'fully digital' exams Some exam boards are already piloting on-screen assessmen
AFTER:  Ofqual and DfE studying 'feasibility' of 'fully digital' exams Some exam boards are already piloting on-screen assessmen

BEFORE: Digital Poverty Alliance A charity whose vision is for everyone to access the life changing benefits that digital brings
AFTER:  Digital Poverty Alliance A charity whose vision is for everyone to access the life changing benefits that digital brings

BEFORE: Lib Dem Munira Wilson, Lib Dem spokesperson for education, is currently drawing up what she says is a "strong education 
AFTER:  Lib Dem Munira Wilson, Lib Dem spokesperson for education, is currently drawing up what she says is a "strong education 

BEFORE: Cutting through the conjecture: How is EdTech really being used in our classrooms? Full post - https://edtech.oii.ox.ac.
AFTER:  Cutting through the conjecture: How is EdTech reall

## 3. Encode metadata features

One-hot encode `item_type` and `org_broad_category` so they can be concatenated with TF-IDF features later.

In [9]:
print("item_type values:")
print(df["item_type"].value_counts())
print(f"\norg_broad_category values:")
print(df["org_broad_category"].value_counts())

item_type values:
item_type
report                 398
news_article           291
government_document    233
academic_article        76
blog_post               69
article                 37
linkedin_post            4
video                    1
Name: count, dtype: int64

org_broad_category values:
org_broad_category
media_sector                             325
government_public_sector                 280
knowledge_mobiliser_think_tank_sector    147
academic_sector                          141
civil_society_nonprofit_sector            96
research_evidence_sector                  73
other_miscellaneous                       30
commercial_private_sector                 12
digital_social_media_platforms             5
Name: count, dtype: int64


In [10]:
# One-hot encode
item_type_dummies = pd.get_dummies(df["item_type"], prefix="item")
org_cat_dummies = pd.get_dummies(df["org_broad_category"], prefix="org")

print(f"item_type columns ({item_type_dummies.shape[1]}): {list(item_type_dummies.columns)}")
print(f"org_broad_category columns ({org_cat_dummies.shape[1]}): {list(org_cat_dummies.columns)}")

item_type columns (8): ['item_academic_article', 'item_article', 'item_blog_post', 'item_government_document', 'item_linkedin_post', 'item_news_article', 'item_report', 'item_video']
org_broad_category columns (9): ['org_academic_sector', 'org_civil_society_nonprofit_sector', 'org_commercial_private_sector', 'org_digital_social_media_platforms', 'org_government_public_sector', 'org_knowledge_mobiliser_think_tank_sector', 'org_media_sector', 'org_other_miscellaneous', 'org_research_evidence_sector']


In [11]:
# Combine into a single metadata feature matrix
metadata_cols = list(item_type_dummies.columns) + list(org_cat_dummies.columns)
df = pd.concat([df, item_type_dummies, org_cat_dummies], axis=1)

print(f"Total metadata features: {len(metadata_cols)}")
print(f"DataFrame shape after encoding: {df.shape}")

Total metadata features: 17
DataFrame shape after encoding: (1109, 27)


## 4. Stratified train/val split

In [12]:
train_df, val_df = train_test_split(
    df,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=df["target"],
)

print(f"Train: {train_df.shape[0]} rows")
print(f"Val:   {val_df.shape[0]} rows")

Train: 942 rows
Val:   167 rows


In [13]:
# Verify stratification preserved class proportions
split_check = pd.DataFrame({
    "train_%": train_df["target"].value_counts(normalize=True).mul(100).round(1),
    "val_%": val_df["target"].value_counts(normalize=True).mul(100).round(1),
    "overall_%": df["target"].value_counts(normalize=True).mul(100).round(1),
})
split_check

,train_%,val_%,overall_%
target,,,
political_environment_key_organisations,21.5,21.6,21.6
what_matters_ed,17.7,18.0,17.8
teacher_rrd,17.5,17.4,17.5
edtech,15.8,15.6,15.8
policy_practice_research,14.4,14.4,14.4
four_nations,13.0,13.2,13.0


## 5. Save

In [14]:
train_df.to_csv(OUTPUT_DIR / "train.csv", index=False)
val_df.to_csv(OUTPUT_DIR / "val.csv", index=False)

print(f"Saved to {OUTPUT_DIR}/")
print(f"  train.csv  — {train_df.shape}")
print(f"  val.csv    — {val_df.shape}")
print(f"\nColumns saved ({len(train_df.columns)}):")
print(list(train_df.columns))

Saved to ../data/modelling/
  train.csv  — (942, 27)
  val.csv    — (167, 27)

Columns saved (27):
['id', 'newsletter_number', 'issue_date', 'link', 'target', 'text', 'organisation', 'org_broad_category', 'item_type', 'text_clean', 'item_academic_article', 'item_article', 'item_blog_post', 'item_government_document', 'item_linkedin_post', 'item_news_article', 'item_report', 'item_video', 'org_academic_sector', 'org_civil_society_nonprofit_sector', 'org_commercial_private_sector', 'org_digital_social_media_platforms', 'org_government_public_sector', 'org_knowledge_mobiliser_think_tank_sector', 'org_media_sector', 'org_other_miscellaneous', 'org_research_evidence_sector']


## Summary

| What | Detail |
|------|--------|
| Input | `eda_cleaned.csv` — 1,109 rows, 6 classes |
| Text cleaning | Removed URLs & emails, kept casing/punctuation |
| Metadata features | One-hot encoded `item_type` + `org_broad_category` |
| Split | 85/15 stratified by `target` (seed=42) |
| Output | `data/modelling/train.csv`, `data/modelling/val.csv` |

**Next:** `03_baseline_tfidf.ipynb` — TF-IDF + logistic regression baseline